In [ ]:
# ==============================================================================
# MỤC 1: SỬ DỤNG THƯ VIỆN (SCIKIT-LEARN) CHO THUẬT TOÁN PHÂN CỤM PHÂN CẤP PHÂN TỤ
# (AGGLOMERATIVE HIERARCHICAL CLUSTERING)
# ==============================================================================

# Import thư viện numpy để xử lý mảng số học và ma trận khoảng cách
import numpy as np
# Import lớp AgglomerativeClustering từ thư viện scikit-learn cho bài toán phân cụm phân cấp phân tụ
from sklearn.cluster import AgglomerativeClustering
# Import các độ đo đánh giá chất lượng phân cụm (Silhouette Score và Adjusted Rand Index)
from sklearn.metrics import silhouette_score, adjusted_rand_score

def run_agglomerative_sklearn(X, n_clusters=3, linkage='single'):
    # Khởi tạo mô hình AgglomerativeClustering:
    # n_clusters: số lượng cụm cuối cùng muốn dừng lại (cắt cây Dendrogram)
    # metric='euclidean': sử dụng khoảng cách Euclid giữa các điểm dữ liệu
    # linkage: phương pháp đo khoảng cách giữa 2 cụm ('single', 'complete', 'average', 'ward')
    model = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric='euclidean',
        linkage=linkage
    )
    
    # Thực hiện phân cụm phân cấp từ dưới lên (bottom-up) và trả về nhãn cụm cho từng điểm dữ liệu
    labels = model.fit_predict(X)
    
    # Trả về mô hình đã huấn luyện và mảng nhãn cụm dự đoán
    return model, labels


In [ ]:
# ==============================================================================
# MỤC 2: TỰ VIẾT CODE THUẬT TOÁN PHÂN CỤM PHÂN CẤP PHÂN TỤ TỪ ĐẦU (FROM SCRATCH)
# ==============================================================================

class CustomAgglomerativeClustering:
    def __init__(self, n_clusters=3, linkage='single'):
        # Khởi tạo các siêu tham số cho thuật toán phân cụm phân cấp phân tụ (Bottom-Up):
        # n_clusters: số cụm mục tiêu cần đạt được sau khi gộp dần các cụm nhỏ
        # linkage: phương pháp liên kết cụm hỗ trợ 'single' (min), 'complete' (max), 'average' (mean)
        self.n_clusters = n_clusters
        self.linkage = linkage
        self.labels_ = None
        self.merge_history_ = []

    def _compute_pairwise_distances(self, X):
        # Tính ma trận khoảng cách Euclid đôi một giữa tất cả các điểm dữ liệu ban đầu
        # Sử dụng broadcasting để tính nhanh ma trận kích thước (n_samples, n_samples)
        diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
        dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
        return dist_matrix

    def _cluster_distance(self, cluster_a, cluster_b, point_dist_matrix):
        # Tính khoảng cách giữa 2 cụm (cluster_a và cluster_b) dựa trên danh sách chỉ số điểm của từng cụm
        # Trích xuất ma trận con chứa khoảng cách giữa mọi cặp điểm (i thuộc A, j thuộc B)
        sub_distances = point_dist_matrix[np.ix_(cluster_a, cluster_b)]
        
        if self.linkage == 'single':
            # Liên kết đơn (Single Linkage): khoảng cách ngắn nhất giữa 2 điểm bất kỳ thuộc 2 cụm
            return np.min(sub_distances)
        elif self.linkage == 'complete':
            # Liên kết đầy đủ (Complete Linkage): khoảng cách xa nhất giữa 2 điểm bất kỳ thuộc 2 cụm
            return np.max(sub_distances)
        elif self.linkage == 'average':
            # Liên kết trung bình (Average Linkage): trung bình cộng khoảng cách mọi cặp điểm giữa 2 cụm
            return np.mean(sub_distances)
        else:
            raise ValueError("Tham số linkage chỉ hỗ trợ: 'single', 'complete', hoặc 'average'")

    def fit_predict(self, X):
        # Lấy tổng số điểm dữ liệu ban đầu
        n_samples = X.shape[0]
        
        # Bước 1: Tính sẵn ma trận khoảng cách Euclid giữa tất cả các cặp điểm dữ liệu
        point_dist_matrix = self._compute_pairwise_distances(X)
        
        # Bước 2: Khởi tạo ban đầu mỗi điểm dữ liệu là một cụm độc lập (tổng cộng n_samples cụm)
        # Dùng dictionary ánh xạ cluster_id -> danh sách các chỉ số điểm thuộc cụm đó
        active_clusters = {i: [i] for i in range(n_samples)}
        
        # Khởi tạo ma trận khoảng cách hiện hành giữa các cụm đang hoạt động
        cluster_dist = point_dist_matrix.copy()
        # Đặt đường chéo chính bằng vô cùng (np.inf) để một cụm không tự gộp với chính nó
        np.fill_diagonal(cluster_dist, np.inf)
        
        # Biến lưu lịch sử các bước gộp cụm (tương tự cấu trúc cây Dendrogram)
        self.merge_history_ = []
        next_cluster_id = n_samples

        # Bước 3: Vòng lặp phân tụ - liên tục gộp 2 cụm gần nhau nhất cho đến khi còn đúng n_clusters cụm
        while len(active_clusters) > self.n_clusters:
            # Tìm vị trí (i, j) có khoảng cách nhỏ nhất trong ma trận khoảng cách cụm hiện tại
            flat_idx = np.argmin(cluster_dist)
            idx_i, idx_j = np.unravel_index(flat_idx, cluster_dist.shape)
            min_dist = cluster_dist[idx_i, idx_j]

            # Đảm bảo idx_i < idx_j để thao tác xóa và cập nhật chỉ số nhất quán
            if idx_i > idx_j:
                idx_i, idx_j = idx_j, idx_i

            # Lấy danh sách ID các cụm đang hoạt động hiện tại
            current_ids = list(active_clusters.keys())
            id_i = current_ids[idx_i]
            id_j = current_ids[idx_j]

            # Gộp các điểm dữ liệu của cụm id_j vào cụm id_i tạo thành cụm mới
            merged_points = active_clusters[id_i] + active_clusters[id_j]
            
            # Ghi lại lịch sử bước gộp (cụm 1, cụm 2, khoảng cách gộp, số lượng điểm sau gộp)
            self.merge_history_.append((id_i, id_j, min_dist, len(merged_points)))

            # Cập nhật dictionary các cụm hoạt động: xóa 2 cụm cũ và thêm cụm mới gộp
            del active_clusters[id_i]
            del active_clusters[id_j]
            active_clusters[next_cluster_id] = merged_points
            next_cluster_id += 1

            # Cập nhật lại ma trận khoảng cách cụm:
            # Xóa hàng/cột của 2 cụm cũ (idx_j trước rồi idx_i để không lệch chỉ số)
            cluster_dist = np.delete(cluster_dist, [idx_i, idx_j], axis=0)
            cluster_dist = np.delete(cluster_dist, [idx_i, idx_j], axis=1)

            # Tính khoảng cách từ cụm mới gộp tới tất cả các cụm còn lại
            remaining_ids = list(active_clusters.keys())[:-1]
            if len(remaining_ids) > 0:
                new_dists = np.array([
                    self._cluster_distance(merged_points, active_clusters[cid], point_dist_matrix)
                    for cid in remaining_ids
                ])
                # Mở rộng thêm 1 hàng và 1 cột vào ma trận khoảng cách cho cụm mới
                cluster_dist = np.vstack([cluster_dist, new_dists])
                new_col = np.append(new_dists, np.inf).reshape(-1, 1)
                cluster_dist = np.hstack([cluster_dist, new_col])

        # Bước 4: Gán nhãn cụm cuối cùng (từ 0 đến n_clusters - 1) cho từng điểm dữ liệu ban đầu
        labels = np.zeros(n_samples, dtype=int)
        for cluster_label, (_, point_indices) in enumerate(active_clusters.items()):
            for pt_idx in point_indices:
                labels[pt_idx] = cluster_label

        self.labels_ = labels
        return labels


In [ ]:
# ==============================================================================
# MỤC 3: TẠO DATA MOCK VÀ KIỂM THỬ SO SÁNH 2 THUẬT TOÁN (THƯ VIỆN VS TỰ VIẾT)
# ==============================================================================

# Import hàm make_blobs để tạo bộ dữ liệu giả lập (mock data) có cấu trúc cụm rõ ràng
from sklearn.datasets import make_blobs

# Bước 1: Tạo dữ liệu giả lập (Mock Data)
# Tạo 150 điểm dữ liệu trong không gian 2 chiều (2 đặc trưng), chia thành 3 cụm thực tế
# (Số lượng mẫu vừa phải giúp trực quan hóa lịch sử gộp và chạy thuật toán phân cấp O(N^3) nhanh chóng)
X, y_true = make_blobs(
    n_samples=150,
    n_features=2,
    centers=3,
    cluster_std=0.9,
    random_state=42
)

print(f"Kích thước tập dữ liệu Mock: {X.shape}, Số cụm thực tế: {len(np.unique(y_true))}\n")

# Thử nghiệm so sánh trên cả 3 phương pháp liên kết phổ biến: 'single', 'complete', 'average'
for linkage_method in ['single', 'complete', 'average']:
    # Bước 2: Chạy thuật toán từ thư viện Scikit-Learn (Mục 1)
    sklearn_model, labels_sklearn = run_agglomerative_sklearn(
        X, n_clusters=3, linkage=linkage_method
    )
    
    # Bước 3: Chạy thuật toán tự viết CustomAgglomerativeClustering (Mục 2)
    custom_model = CustomAgglomerativeClustering(n_clusters=3, linkage=linkage_method)
    labels_custom = custom_model.fit_predict(X)
    
    # Bước 4: Đánh giá và so sánh hiệu năng giữa 2 phương pháp
    # Sử dụng Adjusted Rand Index (ARI) để kiểm tra mức độ trùng khớp cấu trúc cụm bất kể hoán vị nhãn
    ari_match = adjusted_rand_score(labels_sklearn, labels_custom)
    ari_vs_true = adjusted_rand_score(y_true, labels_custom)
    sil_sklearn = silhouette_score(X, labels_sklearn)
    sil_custom = silhouette_score(X, labels_custom)
    
    print(f"=== KẾT QUẢ SO SÁNH VỚI LINKAGE = '{linkage_method.upper()}' ===")
    print(f"Silhouette Score - Thư viện Sklearn  : {sil_sklearn:.4f}")
    print(f"Silhouette Score - Thuật toán tự viết : {sil_custom:.4f}")
    print(f"Adjusted Rand Index (so với nhãn gốc) : {ari_vs_true:.4f}")
    print(f"Độ trùng khớp phân cụm giữa 2 mô hình : {ari_match * 100:.2f}%\n")

# In thử 5 bước gộp cụm đầu tiên và 3 bước gộp cụm cuối cùng từ lịch sử của mô hình tự viết (average linkage)
print("Lịch sử gộp cụm (Dendrogram steps) - Mô hình tự viết (Average Linkage):")
print("3 bước gộp đầu tiên (Cụm A, Cụm B, Khoảng cách, Số điểm mới):")
for step in custom_model.merge_history_[:3]:
    print(f"  Gộp cụm {step[0]:3d} + cụm {step[1]:3d} | Khoảng cách = {step[2]:.4f} | Tổng điểm = {step[3]}")

print("3 bước gộp cuối cùng:")
for step in custom_model.merge_history_[-3:]:
    print(f"  Gộp cụm {step[0]:3d} + cụm {step[1]:3d} | Khoảng cách = {step[2]:.4f} | Tổng điểm = {step[3]}")
